In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import numpy as np
import pickle
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------
# 1. Define SimCLR (must match the saved checkpoint)
# ---------------------------------------------------
class SimCLR(nn.Module):
    def __init__(self, base_encoder):
        super(SimCLR, self).__init__()
        self.encoder = nn.Sequential(*list(base_encoder.children())[:-1])  # remove fc
        dim_mlp = base_encoder.fc.in_features
        self.projector = nn.Sequential(
            nn.Linear(dim_mlp, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, 1)
        z = self.projector(h)
        return F.normalize(z, dim=1)

# ---------------------------------------------------
# 2. Load ResNet18 and pretrained SimCLR weights
# ---------------------------------------------------
base_encoder = models.resnet18(weights=None)
simclr_encoder = SimCLR(base_encoder)

state_dict = torch.load("ssl_encoder.pth", map_location=device)
missing, unexpected = simclr_encoder.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {missing}\nUnexpected keys: {unexpected}")

simclr_encoder = simclr_encoder.to(device)
print("✅ SSL encoder loaded and ready for fine-tuning.")

# ---------------------------------------------------
# 3. Unfreeze most layers for better adaptation
# ---------------------------------------------------
# Unfreeze all except first conv and bn (helps stability)
for name, param in simclr_encoder.encoder.named_parameters():
    if "encoder.0" in name or "encoder.1" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# ---------------------------------------------------
# 4. Data setup for 1024×1024 fine-tuning
# ---------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(root="data/train", transform=train_transform)
train_loader = DataLoader(train_data, batch_size=2, shuffle=True, num_workers=2)

# ---------------------------------------------------
# 5. Attach fine-tuning head (classification)
# ---------------------------------------------------
num_classes = len(train_data.classes)
finetune_head = nn.Linear(128, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    list(finetune_head.parameters()) + 
    [p for p in simclr_encoder.parameters() if p.requires_grad],
    lr=1e-4
)

# ---------------------------------------------------
# 6. Fine-tuning loop
# ---------------------------------------------------
epochs = 30  # recommended for visible improvement
for epoch in range(epochs):
    simclr_encoder.train()
    finetune_head.train()
    running_loss = 0.0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        feats = simclr_encoder(imgs)
        outputs = finetune_head(feats)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

torch.save(simclr_encoder.state_dict(), "ssl_encoder_finetuned.pth")
print("✅ Saved fine-tuned model as ssl_encoder_finetuned.pth")

# ---------------------------------------------------
# 7. Extract features and save to ssl_features.pkl
# ---------------------------------------------------
simclr_encoder.eval()
all_features, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(train_loader, desc="Extracting Fine-Tuned Features"):
        imgs = imgs.to(device)
        feats = simclr_encoder(imgs).cpu().numpy()
        all_features.append(feats)
        all_labels.append(labels.cpu().numpy())

X = np.concatenate(all_features)
y = np.concatenate(all_labels)

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

with open("ssl_features.pkl", "wb") as f:
    pickle.dump({"features": X, "labels": y}, f)

print("✅ Saved fine-tuned features to ssl_features.pkl")


Using device: cpu
Missing keys: []
Unexpected keys: []
✅ SSL encoder loaded and ready for fine-tuning.


Epoch 1/30: 100%|██████████| 500/500 [16:44<00:00,  2.01s/it]


Epoch 1/30 - Loss: 0.6791


Epoch 2/30: 100%|██████████| 500/500 [16:21<00:00,  1.96s/it]


Epoch 2/30 - Loss: 0.6580


...

Epoch 29/30 - Loss: 0.3854


Epoch 30/30: 100%|██████████| 500/500 [15:13<00:00,  1.83s/it]


Epoch 30/30 - Loss: 0.3771
✅ Saved fine-tuned model as ssl_encoder_finetuned.pth


Extracting Fine-Tuned Features: 100%|██████████| 500/500 [05:03<00:00,  1.65it/s]

Feature matrix shape: (1000, 128)
Label vector shape: (1000,)
✅ Saved fine-tuned features to ssl_features.pkl
